# LAB-18 · LoRA fine-tune on a free GPU

Fine-tuning every weight of a model is like reprinting a whole book to fix a few pages. LoRA leaves the book alone and
clips small sticky notes onto some pages: two thin matrices per weight matrix, trained while the original weights stay
frozen. You will teach a small open model (Qwen3-0.6B) one new skill, count exactly how few weights you train, and save
the result as a small adapter file.

~25 minutes · **GPU optional**. On a CPU the training cell runs fewer steps and is slower, but every check still runs;
it prints how long each step took on your device. Run the cells in order, fill the `# TODO` gaps, then use the checks;
starter failures are expected. A complete Solution section follows "try first".

Use a fresh Python runtime. Setup installs pinned library versions in your notebook account if they are missing. The
model downloads from the Hugging Face Hub (about 1.5 GB) and needs no account or API key.

## Setup

Pinned versions keep this notebook reproducible. PyTorch comes from your runtime (hosted notebooks preinstall it).
The cell prints your device and its memory, so you know what you are working with. Hosted services decide which GPU
you get, if any.

In [ ]:
import importlib.metadata as md

PINS = {"transformers": "5.23.0", "peft": "0.22.0", "accelerate": "1.16.0"}
def installed(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
NEED = " ".join(f"{k}=={v}" for k, v in PINS.items() if installed(k) != v)
if NEED:
    # Runs only in your notebook runtime. Restart the runtime if transformers was already imported.
    print("installing", NEED)
    %pip install -q {NEED}


In [ ]:
import json, math, os, random, time
import torch
import transformers, peft
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
torch.manual_seed(0)
random.seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")
else:
    print("No GPU: CPU run with fewer training steps (slower, same checks)")

## Load a small open model

Qwen3-0.6B is small enough to train in 32-bit floats on a CPU or a free GPU, which keeps the numbers simple: 4 bytes
per weight. Count its weights first; this is the number LoRA avoids training.

In [ ]:
MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"weights: {total:,} ({total * 4 / 2**30:.2f} GiB in float32)")

## The skill to teach

The task: read one sentence about a graphics card and answer with exactly one line of JSON, nothing else. The examples
below are generated in the notebook (no download). Two sentences are held out: the model never trains on them, so
they test whether it learned the pattern rather than memorizing answers.

In [ ]:
CARDS = [("RTX 3060", 12), ("RTX 4060", 8), ("RTX 4060 Ti", 16), ("RTX 4070", 12), ("RTX 4080", 16),
         ("RTX 4090", 24), ("RTX 5090", 32), ("RTX 3090", 24), ("RTX 2060", 6), ("GTX 1080 Ti", 11),
         ("T4", 16), ("L4", 24), ("A100", 80), ("RX 7900 XTX", 24), ("Arc A770", 16), ("RX 6700 XT", 12)]
TEMPLATES = ["I just bought an {g}, it has {m} GB of memory.", "My desktop runs an {g} with {m} GB.",
             "Can a {m} GB {g} run this model?", "Our team shares one {g} ({m} GB of VRAM).",
             "Selling my {g}, {m} GB, barely used.", "The cloud box gives me an {g} with {m} GB."]

def example(gpu, mem, k):
    sentence = TEMPLATES[k % len(TEMPLATES)].format(g=gpu, m=mem)
    answer = json.dumps({"gpu": gpu, "memory_gb": mem})
    return [{"role": "user", "content": "Extract the GPU and its memory as JSON: " + sentence},
            {"role": "assistant", "content": answer}]

TRAIN = [example(g, m, k) for k, (g, m) in enumerate(CARDS)] + [example(g, m, k + 3) for k, (g, m) in enumerate(CARDS)]
TEST = [example("H100", 80, 1), example("RTX 3080", 10, 3)]
print(len(TRAIN), "training chats; first one:")
print(TRAIN[0])

In [ ]:
def ask(messages, max_new_tokens=40):
    prompt = tok.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def score(chats):
    ok = 0
    for chat in chats:
        reply = ask(chat)
        try:
            good = json.loads(reply) == json.loads(chat[1]["content"])
        except json.JSONDecodeError:
            good = False
        ok += good
        print("PASS" if good else "FAIL", repr(reply[:70]))
    return ok

model.eval()
print("before training:", score(TEST), "of", len(TEST))

## Add LoRA adapters

For a weight matrix with `d_in` inputs and `d_out` outputs, LoRA adds `A` (`r × d_in`) and `B` (`d_out × r`), so it
trains `r × (d_in + d_out)` numbers instead of `d_in × d_out`. `B` starts at zero, so the model starts out unchanged.

**Try first:** finish `lora_params`. The check compares it with the shapes PEFT really created.

In [ ]:
def lora_params(d_in, d_out, r):
    # TODO: number of trainable numbers LoRA adds to one d_out x d_in weight matrix
    return 0

In [ ]:
R = 16
config = LoraConfig(r=R, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, config)
model.print_trainable_parameters()

def check(label, test):
    try:
        test()
    except Exception as error:
        print(f"FAIL {label}: {type(error).__name__}: {error}")
        return False
    print(f"PASS {label}")
    return True

def check_count():
    q = model.base_model.model.model.layers[0].self_attn.q_proj
    d_out, d_in = q.base_layer.weight.shape
    real = q.lora_A["default"].weight.numel() + q.lora_B["default"].weight.numel()
    got = lora_params(d_in, d_out, R)
    torch.testing.assert_close(torch.tensor(got), torch.tensor(real), msg=f"q_proj is {d_out} x {d_in}: expected {real}, got {got}")
check("LoRA count for one matrix", check_count)

## Train only the adapters

Each chat becomes one training sequence through the model's own chat template. The loss counts only the answer
tokens (the prompt is masked with `-100`), and the optimizer sees only the LoRA weights. The cell prints the loss as it
falls, and how long each step took on your device.

In [ ]:
def encode(chat):
    prompt = tok.apply_chat_template(chat[:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    full = tok.apply_chat_template(chat, tokenize=False, enable_thinking=False)
    assert full.startswith(prompt)
    p, f = tok(prompt)["input_ids"], tok(full)["input_ids"]
    return f, [-100] * len(p) + f[len(p):]

def batch_of(chats):
    rows = [encode(c) for c in chats]
    n = max(len(ids) for ids, _ in rows)
    pad = tok.pad_token_id
    ids = torch.tensor([i + [pad] * (n - len(i)) for i, _ in rows])
    labels = torch.tensor([l + [-100] * (n - len(l)) for _, l in rows])
    mask = torch.tensor([[1] * len(i) + [0] * (n - len(i)) for i, _ in rows])
    return ids.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)

STEPS = 60 if HAS_GPU else 30
BATCH = 8
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)
model.train()
t0 = time.time()
for step in range(1, STEPS + 1):
    ids, labels, mask = batch_of(random.sample(TRAIN, BATCH))
    loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
    loss.backward()
    opt.step()
    opt.zero_grad()
    if step % 5 == 0 or step == 1:
        print(f"step {step:3d}  loss {loss.item():.3f}")
print(f"{STEPS} steps, {(time.time() - t0) / STEPS:.2f} s per step on {DEVICE} (your run)")

In [ ]:
model.eval()
print("after training:", score(TEST), "of", len(TEST))

## Save the adapter

Only the LoRA matrices are saved. The base model stays as downloaded; anyone with the same base model can load your
adapter on top of it.

In [ ]:
model.save_pretrained("lora-adapter")
for name in sorted(os.listdir("lora-adapter")):
    print(f"{os.path.getsize(os.path.join('lora-adapter', name)):>12,} B  {name}")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{trainable:,} trainable numbers x 4 bytes = {trainable * 4:,} B")

## Why big models run out of memory

Training keeps more than the weights. For every trainable number, PyTorch also stores its gradient, and the AdamW
optimizer keeps two running averages. In 32-bit floats that is 4 + 4 + 8 = 16 bytes per trainable number, plus 4
bytes per frozen weight, plus activations (not counted here). The estimate below uses your model's real counts.

In [ ]:
def gib(n_bytes):
    return n_bytes / 2**30

frozen = total
print(f"full fine-tune: {gib(total * 16):6.2f} GiB  (every weight trainable, 16 bytes each)")
print(f"LoRA r={R}:      {gib(frozen * 4 + trainable * 16):6.2f} GiB  (frozen weights 4 bytes, adapters 16 bytes)")
print(f"an 8B model's weights alone in 16-bit: {gib(8e9 * 2):.1f} GiB")

An 8-billion-parameter model needs about 15 GiB just to hold its weights in 16-bit floats, before any training
state. That is why people fine-tune big models on small GPUs with the frozen base stored in 4 bits (QLoRA), or pick a
smaller model.

## Solution

Try first. Run all continues here, replaces the starter function, and checks it again.

In [ ]:
def lora_params(d_in, d_out, r):
    return r * d_in + d_out * r   # A is r x d_in, B is d_out x r

assert check("LoRA count for one matrix (solution)", check_count)

Back to [LAB-18 in the Tile & Error Academy](https://tile-and-error-academy.vercel.app/#lab/lab-18), or see all
[Academy labs](https://tile-and-error-academy.vercel.app/#labs).